In [16]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure the saved_models directory exists in the parent folder
os.makedirs("../frontend/saved_models", exist_ok=True)
print("Environment ready.")

Environment ready.


In [17]:
import re
import pandas as pd
import numpy as np

# 1. Load the actual dataset
file_path = r"C:\Users\pipal\OneDrive\Desktop\career_intelligence_system\scripts\data\raw\resume_data.csv"

try:
    raw_df = pd.read_csv(file_path)
    print(f"✅ Successfully loaded {len(raw_df)} resumes from dataset.")
except FileNotFoundError:
    print(f"❌ Error: Could not find the file at {file_path}")
    raw_df = pd.DataFrame({"Resume": ["Python developer with 3 years experience. B.Tech degree. 5 projects done. AWS certified."] * 500})

# Find the text column
text_col = None
for col in raw_df.columns:
    if 'resume' in col.lower() or 'text' in col.lower():
        text_col = col
        break

if not text_col:
    print("Warning: Could not automatically detect the text column. Assuming column index 0 or 1.")
    text_col = raw_df.columns[1] if len(raw_df.columns) > 1 else raw_df.columns[0]

# 2. Extract Numerical Features
print("Extracting features from raw resume text... (This might take a few seconds)")

extracted_data = []
for text in raw_df[text_col].astype(str):
    text_lower = text.lower()
    
    exp = min(len(text_lower.split()) / 300.0, 10.0)
    if "senior" in text_lower or "lead" in text_lower: exp = max(exp, 5.0)
    projects = min(len(re.findall(r"\bproject\w*", text_lower)), 15)
    
    if any(w in text_lower for w in ["phd", "master", "m.tech", "mba"]): edu = 2
    elif any(w in text_lower for w in ["bachelor", "b.tech", "bsc", "degree"]): edu = 1
    else: edu = 0
        
    certs = min(len(re.findall(r"\bcertif\w+", text_lower)), 10)
    
    tech_keywords = ['python', 'java', 'sql', 'aws', 'docker', 'react', 'linux', 'cloud', 'data', 'api']
    keyword_hits = sum(1 for kw in tech_keywords if kw in text_lower)
    
    base_fit = min((keyword_hits / len(tech_keywords)) * 100 + 40, 95)
    domain_fit = np.clip(np.random.normal(base_fit, 10), 0, 100)
    gap = np.clip(100 - domain_fit + np.random.normal(0, 5), 0, 100)
    
    extracted_data.append({
        'experience_years': round(exp, 1),
        'project_count': int(projects),
        'education_score': edu,
        'certification_count': int(certs),
        'domain_fit_score': round(domain_fit, 1),
        'gap_percentage': round(gap, 1)
    })

df = pd.DataFrame(extracted_data)

# 3. Generate the Target Variable (Shortlisted)
hr_scores = (
    (df['experience_years'] * 5) + 
    (df['project_count'] * 2) + 
    (df['education_score'] * 8) + 
    (df['certification_count'] * 3) + 
    (df['domain_fit_score'] * 0.6) - 
    (df['gap_percentage'] * 0.5)
)

# DYNAMIC FIX: Find the median score of your specific dataset and center the curve around it
# This ensures that roughly 30-40% of resumes get shortlisted, giving the AI healthy data to learn from.
center_score = hr_scores.median() + 2 
probabilities = 1 / (1 + np.exp(-(hr_scores - center_score) / 5))

# Assign 1 (Shortlisted) or 0 (Rejected)
df['shortlisted'] = (np.random.rand(len(df)) < probabilities).astype(int)

print(f"✅ Data processing complete. Final dataset shape: {df.shape}")
print(f"🎯 Shortlisting rate adjusted for model training: {df['shortlisted'].mean()*100:.1f}%")

# DISPLAY FIX: Using print() to bypass the pandas HTML rendering bug
print("\nFirst 5 rows of processed data:")
print(df.head().to_string())

✅ Successfully loaded 9544 resumes from dataset.
Extracting features from raw resume text... (This might take a few seconds)
✅ Data processing complete. Final dataset shape: (9544, 7)
🎯 Shortlisting rate adjusted for model training: 45.4%

First 5 rows of processed data:
   experience_years  project_count  education_score  certification_count  domain_fit_score  gap_percentage  shortlisted
0               0.1              0                0                    0              65.5            30.8            1
1               0.1              0                0                    0              42.2            62.9            0
2               0.0              0                0                    0              52.9            52.5            1
3               0.1              0                0                    0              48.6            45.3            1
4               0.1              0                0                    0              29.1            75.3            0


In [18]:
# Define features (X) and target (y)
X = df.drop('shortlisted', axis=1)
y = df['shortlisted']

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features (Standardization is important for SHAP and some ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled and split successfully.")

Data scaled and split successfully.


In [19]:
# Initialize and train the Random Forest Classifier
# We use 200 trees and limit depth to prevent overfitting and make SHAP run faster
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test_scaled)
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 0.8119434258774227

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.84      0.83      1042
           1       0.80      0.78      0.79       867

    accuracy                           0.81      1909
   macro avg       0.81      0.81      0.81      1909
weighted avg       0.81      0.81      0.81      1909



In [20]:
# The frontend app.py attempts to load a vectorizer.pkl. 
# While our Random Forest uses numerical features, we will create a basic 
# TF-IDF vectorizer here for text extraction purposes in your NLP Processor.

dummy_corpus = [
    "python java html css javascript react frontend backend",
    "machine learning data science artificial intelligence sql python",
    "aws azure docker kubernetes devops ci cd cloud",
    "cybersecurity network firewall security penetration testing"
]
vectorizer = TfidfVectorizer(max_features=500)
vectorizer.fit(dummy_corpus)

# --- Save the Models ---
rf_path = "../frontend/saved_models/rf_classifier.pkl"
scaler_path = "../frontend/saved_models/scaler.pkl"
vec_path = "../frontend/saved_models/vectorizer.pkl"

with open(rf_path, "wb") as f:
    pickle.dump(rf_model, f)

with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
    
with open(vec_path, "wb") as f:
    pickle.dump(vectorizer, f)

print(f"✅ Success! Models saved to:")
print(f" - {rf_path}")
print(f" - {scaler_path}")
print(f" - {vec_path}")

✅ Success! Models saved to:
 - ../frontend/saved_models/rf_classifier.pkl
 - ../frontend/saved_models/scaler.pkl
 - ../frontend/saved_models/vectorizer.pkl
